In [2]:
# Cell 1 – FIRST, before any app imports
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Explicit path; Jupyter project root is usually /app or /code
for p in [Path("/app/.env"), Path("/code/.env"), Path.cwd() / ".env"]:
    if p.exists():
        load_dotenv(p)
        print(f"Loaded from {p}")
        break
else:
    print("No .env found")

# Sanity check
print("HUGGINGFACE_API_TOKEN set:", bool(os.getenv("HUGGINGFACE_API_TOKEN")))
print("OPENAI_API_TOKEN set:", bool(os.getenv("OPENAI_API_TOKEN")))
# override DATABASE_URL to look at local
os.environ["DATABASE_URL"] = "postgresql+psycopg2://badger:badgerpass@db:5432/badgerdb"
sys.path.append(str(Path().resolve().parent))
from app.db import engine, SessionLocal#from app.models import User, Document, VaultMembership
from app.topics import recluster_vaults_with_tree #as rct
from uuid import UUID

Loaded from /app/.env
HUGGINGFACE_API_TOKEN set: True
OPENAI_API_TOKEN set: False


In [3]:
import numpy as np
#%pip install pandas
#!{sys.executable} -m pip install pandas
#%pip install matplotlib
from matplotlib import pyplot as plt
import pandas as pd


In [52]:
qry = """SELECT cs.*, ne.payload -> 'title' AS title
, payload -> 'key_topics' ->> 1 AS first_key_topic
FROM graph.get_compressed_subtree(
  (SELECT array_agg(DISTINCT orchard_id) FROM graph.v_orchard tz 
where tree_id = '75e5abf9-bf3a-421c-872a-e3a5b81d2eec'
and edge_ix =1)) cs
left join graph.node_enrichment ne on ne.doc_id=cs.branch_id
"""
df_tree = pd.read_sql(qry,engine)
df_tree['l_node_ix']=df_tree['l_node_ix']-1
df_tree['r_node_ix']=df_tree['r_node_ix']-1

df_tree.head()

,tree_id,branch_id,branch_ix,l_node_ix,r_node_ix,l_node_id,r_node_id,l_edge_ix,r_edge_ix,doc_count,coph_distance,title,first_key_topic
0,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,25a5d008-6902-497d-a528-7bbb9aa67b9f,165,1,0,b61e0cb4-6fb7-4a3e-a91c-d91709496299,33aff4d2-9d89-4c9b-878d-e754d0d770a0,1,1,2,0.086124,Comprehensive Overview of Mouth Taping for Sle...,Nasal vs mouth breathing benefits
1,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,dc1a0d28-bb29-4b38-8626-21a023bffa67,166,3,2,2d91e5e8-548a-4fb4-841d-93b50c94dc28,cc01d4b1-3430-4703-a8f2-d298ca660078,1,1,2,0.097535,US Military Operation in Venezuela: Capture of...,Capture and indictment of Nicolás Maduro and C...
2,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,ecee6fae-8a38-4728-98c2-67c9041a218e,167,5,4,d4008893-86cc-4ddd-b049-707548cf1363,56729489-b340-4abc-9150-b370d8d430a8,1,1,2,0.110823,Voice & Sight Tag Program in Boulder: Comprehe...,Online registration and renewal process
3,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,78b9ac59-893a-4b03-9d4e-3cfd76e10c5d,168,6,7,17de6ee5-1a1e-498d-bc9e-4f9d58cb6270,0f49ae76-b014-4b22-bf26-d227a4360330,1,1,2,0.111063,Packaging and Publishing Chrome Extensions,Using chrome://extensions
4,75e5abf9-bf3a-421c-872a-e3a5b81d2eec,b07758e1-35b9-4c1c-a721-056d6a9deb20,169,164,8,25a5d008-6902-497d-a528-7bbb9aa67b9f,4d6b18e9-b18a-48d0-a6f7-9e7796d1d1d7,2,1,3,0.116608,Comprehensive Guide and Product Review of Mout...,Nasal vs mouth breathing benefits


In [53]:
#df_tree['l_node_ix']=df_tree['l_node_ix']-1
#df_tree['r_node_ix']=df_tree['r_node_ix']-1
key_topic = df_tree['title'].to_list()
Z = df_tree[['l_node_ix', 'r_node_ix', 'coph_distance', 'doc_count']].to_numpy()
Z.shape

(163, 4)

In [46]:
from scipy.cluster.hierarchy import fcluster

n_clusters = 5
labels = fcluster(Z, n_clusters, criterion="maxclust")


In [7]:
unique_labels, counts = np.unique(labels, return_counts=True)
print(f"Top-level clusters (5): {list(zip(unique_labels, counts))}")
for l,c in zip(unique_labels, counts):
    print(f"{l}: {c}")

Top-level clusters (5): [(np.int32(1), np.int64(6)), (np.int32(2), np.int64(4)), (np.int32(3), np.int64(2)), (np.int32(4), np.int64(151)), (np.int32(5), np.int64(1))]
1: 6
2: 4
3: 2
4: 151
5: 1


In [42]:
import numpy as np
from scipy.cluster.hierarchy import fcluster, leaders


class LinkageClusterPacker:
    def __init__(
        self,
        Z,
        min_nodes=3,
        max_nodes=7,
        target_nodes=5,
        num_splits=10,
        max_depth=4,
        doc_ids=None,
        verbose=True,
        root_id="root",
        branch_labels=None,
    ):
        self.Z = np.asarray(Z)
        self.min_nodes = int(min_nodes)
        self.max_nodes = int(max_nodes)
        self.target_nodes = int(target_nodes)
        self.num_splits = int(num_splits)
        self.max_depth = int(max_depth)
        self.verbose = bool(verbose)
        self.root_id = root_id

        self.n_docs = self.Z.shape[0] + 1
        self.doc_ids = np.arange(self.n_docs) if doc_ids is None else np.asarray(doc_ids)

        self.cut_values = None
        self.labels_by_cut = {}
        self.top_cut = None
        self.top_labels = None
        self.messages = []
        self.branch_labels = branch_labels
        self.node_label_map = {}
        
        if branch_labels is not None:
            if len(branch_labels) != self.Z.shape[0]:
                raise ValueError("branch_labels must have length Z.shape[0]")
            self.node_label_map = {
                self.n_docs + i: branch_labels[i]
                for i in range(self.Z.shape[0])
            }

    # -----------------------------
    # branch label preparation
    # -----------------------------

    def _label_for_node_id(self, node_id, fallback=None):
        if node_id is None:
            return fallback
        return self.node_label_map.get(int(node_id), fallback)
    
    
    def _leader_map(self, labels):
        """
        For a flat cluster assignment vector `labels`, return:
        flat cluster id -> linkage leader node id
        """
        L, M = leaders(self.Z, labels)
        return {int(flat_id): int(leader_id) for leader_id, flat_id in zip(L, M)}
    # -----------------------------
    # Cut preparation
    # -----------------------------
    def _unique_cut_grid(self):
        heights = np.unique(self.Z[:, 2])
        if len(heights) <= self.num_splits:
            return heights

        idx = np.linspace(0, len(heights) - 1, self.num_splits).round().astype(int)
        return np.unique(heights[idx])

    def precalculate_cuts(self):
        self.cut_values = self._unique_cut_grid()
        self.labels_by_cut = {
            float(cut): fcluster(self.Z, t=float(cut), criterion="distance")
            for cut in self.cut_values
        }
        return self.labels_by_cut

    # -----------------------------
    # Grouping and scoring
    # -----------------------------
    def _groups_within_mask(self, labels, mask):
        masked_idx = np.where(mask)[0]
        masked_labels = labels[mask]

        groups = {}
        for lab in np.unique(masked_labels):
            local_idx = masked_idx[masked_labels == lab]
            groups[int(lab)] = local_idx
        return groups

    def _cut_summary_for_mask(self, labels, mask):
        groups = self._groups_within_mask(labels, mask)
        leader_map = self._leader_map(labels)
    
        singleton_docs = []
        nonsingleton = {}
    
        for flat_label, idx in groups.items():
            leader_id = leader_map.get(int(flat_label))
    
            if len(idx) == 1:
                singleton_docs.append(int(idx[0]))
            else:
                nonsingleton[int(flat_label)] = {
                    "doc_indices": idx,
                    "leader_id": leader_id,
                    "display_label": self._label_for_node_id(
                        leader_id,
                        fallback=f"cluster_{leader_id}" if leader_id is not None else f"cluster_{flat_label}"
                    ),
                }
    
        singleton_docs = np.array(sorted(singleton_docs), dtype=int)
        child_sizes = np.array(
            [len(v["doc_indices"]) for v in nonsingleton.values()],
            dtype=int
        )
    
        visible_n_docs = len(singleton_docs) + len(nonsingleton)
    
        return {
            "groups": groups,
            "singleton_docs": singleton_docs,
            "nonsingleton": nonsingleton,
            "child_sizes": child_sizes,
            "n_singletons": len(singleton_docs),
            "n_children": len(nonsingleton),
            "visible_n_docs": int(visible_n_docs),
        }

    def _score_cut(self, summary):
        visible_n_docs = summary["visible_n_docs"]
        n_children = summary["n_children"]

        if visible_n_docs <= 1:
            return None

        score = abs(visible_n_docs - self.target_nodes)

        if visible_n_docs < self.min_nodes:
            score += 3 * (self.min_nodes - visible_n_docs)

        if visible_n_docs > self.max_nodes:
            score += 3 * (visible_n_docs - self.max_nodes)

        if n_children == 0:
            score += 1000

        return float(score)

    # -----------------------------
    # Top-level selection
    # -----------------------------
    def _choose_top_level(self):
        labels = fcluster(self.Z, t=self.target_nodes, criterion="maxclust")
        self.top_cut = 1.5
        self.top_labels = labels
        root_summary = self._cut_summary_for_mask(
            labels,
            np.ones(self.n_docs, dtype=bool)
        )
        return {
            "cut": 1.5,
            "labels": labels,
            "summary": root_summary,
            "score": 1.5,
        }

    def _choose_subcut_for_mask(self, parent_mask, parent_cut):
        lower_cuts = [float(c) for c in self.cut_values if float(c) < float(parent_cut)]
        if not lower_cuts:
            return None

        best = None

        for cut in lower_cuts:
            labels = self.labels_by_cut[cut]
            summary = self._cut_summary_for_mask(labels, parent_mask)
            score = self._score_cut(summary)

            if score is None:
                continue

            candidate = {
                "cut": cut,
                "labels": labels,
                "summary": summary,
                "score": score,
            }

            if best is None:
                best = candidate
            elif candidate["score"] < best["score"]:
                best = candidate
            elif candidate["score"] == best["score"] and candidate["cut"] > best["cut"]:
                best = candidate

        return best

    # -----------------------------
    # Utilities
    # -----------------------------
    def _warn_oversized_bottom_cluster(self, cluster_id, n_docs):
        if n_docs > self.max_nodes:
            msg = (
                f"Bottom cluster {cluster_id} has {n_docs} docs, "
                f"which is larger than max_nodes={self.max_nodes}. "
                f"Consider a larger target_nodes or larger max_depth."
            )
            self.messages.append(msg)
            if self.verbose:
                print(msg)

    def _make_node(self, cluster_id, level, cut_value, doc_indices, display_label=None, leader_id=None):
        doc_indices = np.asarray(doc_indices, dtype=int)
        return {
            "cluster_id": cluster_id,
            "display_label": display_label if display_label is not None else cluster_id,
            "leader_id": None if leader_id is None else int(leader_id),
            "level": int(level),
            "cut_value": None if cut_value is None else float(cut_value),
            "n_total_docs": int(len(doc_indices)),
            "visible_n_docs": int(len(doc_indices)),
            "documents": [],
            "document_indices": [],
            "children": [],
        }

    # -----------------------------
    # Main builder
    # -----------------------------
    def build_payload(self):
        if not self.labels_by_cut:
            self.precalculate_cuts()

        self.messages = []

        top = self._choose_top_level()
        root_mask = np.ones(self.n_docs, dtype=bool)
        root_doc_indices = np.where(root_mask)[0]

        payload = {
            "meta": {
                "n_docs": int(self.n_docs),
                "min_nodes": int(self.min_nodes),
                "max_nodes": int(self.max_nodes),
                "target_nodes": int(self.target_nodes),
                "num_splits": int(self.num_splits),
                "max_depth": int(self.max_depth),
                "top_cut": float(self.top_cut),
                "candidate_cuts": [float(c) for c in self.cut_values],
            },
            "messages": [],
            "root": self._make_node(
                cluster_id=self.root_id,
                display_label=self.root_id,
                leader_id=None,
                level=0,
                cut_value=self.top_cut,
                doc_indices=root_doc_indices,
            ),
        }

        root_node = payload["root"]
        root_summary = top["summary"]

        root_singletons = root_summary["singleton_docs"]
        root_nonsingleton = root_summary["nonsingleton"]

        root_node["documents"] = self.doc_ids[root_singletons].tolist()
        root_node["document_indices"] = root_singletons.tolist()
        root_node["visible_n_docs"] = int(root_summary["visible_n_docs"])

        stack = []

        child_items = sorted(
            root_nonsingleton.items(),
            key=lambda kv: np.min(kv[1]["doc_indices"])
        )
        
        for child_num, (_, child_info) in reversed(list(enumerate(child_items, start=1))):
            child_doc_indices = child_info["doc_indices"]
            child_mask = np.zeros(self.n_docs, dtype=bool)
            child_mask[child_doc_indices] = True
        
            stack.append({
                "mask": child_mask,
                "cut_value": self.top_cut,
                "cluster_id": f"{self.root_id}.{child_num}",
                "display_label": child_info["display_label"],
                "leader_id": child_info["leader_id"],
                "level": 1,
                "parent_children": root_node["children"],
            })

        while stack:
            item = stack.pop()

            mask = item["mask"]
            cut_value = item["cut_value"]
            cluster_id = item["cluster_id"]
            display_label = item.get("display_label", cluster_id)
            leader_id = item.get("leader_id")
            level = item["level"]
            parent_children = item["parent_children"]
            
            doc_indices = np.where(mask)[0]
            node = self._make_node(
                cluster_id=cluster_id,
                display_label=display_label,
                leader_id=leader_id,
                level=level,
                cut_value=cut_value,
                doc_indices=doc_indices,
            )

            if len(doc_indices) <= self.target_nodes:
                node["documents"] = self.doc_ids[doc_indices].tolist()
                node["document_indices"] = doc_indices.tolist()
                node["visible_n_docs"] = int(len(doc_indices))
                parent_children.append(node)
                continue

            if level >= self.max_depth:
                node["documents"] = self.doc_ids[doc_indices].tolist()
                node["document_indices"] = doc_indices.tolist()
                node["visible_n_docs"] = int(len(doc_indices))
                self._warn_oversized_bottom_cluster(cluster_id, len(doc_indices))
                parent_children.append(node)
                continue

            subcut = self._choose_subcut_for_mask(mask, cut_value)

            if subcut is None:
                node["documents"] = self.doc_ids[doc_indices].tolist()
                node["document_indices"] = doc_indices.tolist()
                node["visible_n_docs"] = int(len(doc_indices))
                self._warn_oversized_bottom_cluster(cluster_id, len(doc_indices))
                parent_children.append(node)
                continue

            summary = subcut["summary"]
            singleton_docs = summary["singleton_docs"]
            nonsingleton = summary["nonsingleton"]

            if len(nonsingleton) == 0:
                node["documents"] = self.doc_ids[doc_indices].tolist()
                node["document_indices"] = doc_indices.tolist()
                node["visible_n_docs"] = int(len(doc_indices))
                self._warn_oversized_bottom_cluster(cluster_id, len(doc_indices))
                parent_children.append(node)
                continue

            node["documents"] = self.doc_ids[singleton_docs].tolist()
            node["document_indices"] = singleton_docs.tolist()
            node["visible_n_docs"] = int(summary["visible_n_docs"])

            parent_children.append(node)

            child_items = sorted(
                nonsingleton.items(),
                key=lambda kv: np.min(kv[1]["doc_indices"])
            )
            
            for child_num, (_, child_info) in reversed(list(enumerate(child_items, start=1))):
                child_doc_indices = child_info["doc_indices"]
                child_mask = np.zeros(self.n_docs, dtype=bool)
                child_mask[child_doc_indices] = True
            
                stack.append({
                    "mask": child_mask,
                    "cut_value": subcut["cut"],
                    "cluster_id": f"{cluster_id}.{child_num}",
                    "display_label": child_info["display_label"],
                    "leader_id": child_info["leader_id"],
                    "level": level + 1,
                    "parent_children": node["children"],
                })

        payload["messages"] = self.messages.copy()
        return payload

In [9]:
from IPython.display import display, HTML
import html

def render_cluster_tree(payload, max_docs_shown=12, expand_to_level=2):
    styles = """
    <style>
      .cluster-tree {
        font-family: Inter, system-ui, -apple-system, Segoe UI, Roboto, sans-serif;
        font-size: 14px;
        line-height: 1.45;
        color: #222;
      }

      .cluster-tree details {
        margin-left: 14px;
        border-left: 1px solid #d9d9d9;
        padding-left: 10px;
      }

      .cluster-tree summary {
        cursor: pointer;
        list-style: none;
        margin: 4px 0;
      }

      .cluster-tree summary::-webkit-details-marker {
        display: none;
      }

      .cluster-tree .node-label {
        display: inline-block;
        background: #f6f8fa;
        border: 1px solid #d0d7de;
        border-radius: 8px;
        padding: 4px 8px;
        margin: 2px 0;
        font-weight: 600;
      }

      .cluster-tree .meta {
        color: #666;
        margin-left: 8px;
        font-size: 12px;
      }

      .cluster-tree .docs {
        margin: 6px 0 8px 0;
        padding-left: 8px;
      }

      .cluster-tree .docs-title {
        color: #666;
        font-size: 12px;
        margin-bottom: 4px;
      }

      .cluster-tree .doc {
        display: inline-block;
        background: #eef6ff;
        border: 1px solid #c7def7;
        border-radius: 999px;
        padding: 2px 8px;
        margin: 2px 4px 2px 0;
        font-size: 12px;
      }

      .cluster-tree .more {
        color: #777;
        font-size: 12px;
        margin-left: 4px;
      }

      .cluster-tree .messages {
        background: #fff8e1;
        border: 1px solid #f0d98a;
        border-radius: 8px;
        padding: 10px 12px;
        margin-bottom: 12px;
      }

      .cluster-tree .messages ul {
        margin: 6px 0 0 18px;
      }

      .cluster-tree .root-meta {
        background: #f6f8fa;
        border: 1px solid #d0d7de;
        border-radius: 8px;
        padding: 10px 12px;
        margin-bottom: 12px;
      }

      .cluster-tree .empty {
        color: #999;
        font-size: 12px;
        font-style: italic;
        margin-left: 8px;
      }
    </style>
    """

    def docs_html(doc_ids):
        if not doc_ids:
            return '<div class="empty">No direct leaf documents</div>'

        shown = doc_ids[:max_docs_shown]
        parts = ['<div class="docs"><div class="docs-title">Leaf documents</div>']
        for d in shown:
            parts.append(f'<span class="doc">{html.escape(str(d))}</span>')
        if len(doc_ids) > max_docs_shown:
            parts.append(f'<span class="more">+{len(doc_ids) - max_docs_shown} more</span>')
        parts.append('</div>')
        return "".join(parts)

    def node_html(node, depth=0):
        open_attr = " open" if depth < expand_to_level else ""

        label = (
            f'<span class="node-label">{html.escape(str(node["cluster_id"]))}</span>'
            f'<span class="meta">'
            f'level={node["level"]} | '
            f'n_total={node["n_total_docs"]} | '
            f'visible={node["visible_n_docs"]} | '
            f'cut={node["cut_value"]}'
            f'</span>'
        )

        children = node.get("children", [])
        documents = node.get("documents", [])

        body = docs_html(documents)
        for child in children:
            body += node_html(child, depth + 1)

        return f"<details{open_attr}><summary>{label}</summary>{body}</details>"

    meta = payload.get("meta", {})
    meta_html = f"""
    <div class="root-meta">
      <b>Cluster Payload</b><br>
      n_docs={meta.get("n_docs")} |
      target_nodes={meta.get("target_nodes")} |
      min_nodes={meta.get("min_nodes")} |
      max_nodes={meta.get("max_nodes")} |
      max_depth={meta.get("max_depth")} |
      top_cut={meta.get("top_cut")}
    </div>
    """

    messages = payload.get("messages", [])
    msg_html = ""
    if messages:
        msg_html = '<div class="messages"><b>Messages</b><ul>'
        for m in messages:
            msg_html += f"<li>{html.escape(str(m))}</li>"
        msg_html += "</ul></div>"

    root = payload.get("root")
    if root is None:
        final_html = (
            styles
            + '<div class="cluster-tree">'
            + meta_html
            + msg_html
            + '<div class="messages"><b>Error</b><ul><li>Payload is missing a root node.</li></ul></div>'
            + '</div>'
        )
        return display(HTML(final_html))

    tree_html = node_html(root, depth=0)
    final_html = f'{styles}<div class="cluster-tree">{meta_html}{msg_html}{tree_html}</div>'

    display(HTML(final_html))

In [54]:

# --- 5. Run the packer on synthetic data ---
packer = LinkageClusterPacker(
    Z=Z,
    min_nodes=3,
    max_nodes=9,
    target_nodes=7,
    num_splits=20,
    max_depth=5,
    doc_ids=[f"doc_{i}" for i in range(Z.shape[0] + 1)],
    verbose=True,
    branch_labels=key_topic,
)
payload = packer.build_payload()

Bottom cluster root.1.1.2.1.1 has 15 docs, which is larger than max_nodes=9. Consider a larger target_nodes or larger max_depth.
Bottom cluster root.1.1.3.1.1 has 11 docs, which is larger than max_nodes=9. Consider a larger target_nodes or larger max_depth.
Bottom cluster root.1.1.3.1.2 has 15 docs, which is larger than max_nodes=9. Consider a larger target_nodes or larger max_depth.


In [11]:
render_cluster_tree(payload)

In [12]:
payload.keys()

dict_keys(['meta', 'messages', 'root'])

In [13]:
n_clusters = 5
labels = fcluster(Z, n_clusters, criterion="maxclust")


In [14]:
set(labels)

{np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5)}

In [99]:
#import pandas as pd
import plotly.graph_objects as go

#import numpy as np

import textwrap

def wrap_treemap_label(s, width=12):
    if s is None:
        return ""
    parts = str(s).split()
    if not parts:
        return str(s)
    return "<br>".join(textwrap.wrap(str(s), width=width, break_long_words=False))

def wrap_treemap_label_by_level(s, level, top_width=56, mid_width=16, deep_width=12):
    if s is None:
        return ""
    width = top_width if level <= 1 else mid_width if level == 2 else deep_width
    return "<br>".join(textwrap.wrap(str(s), width=width, break_long_words=False))
    
def payload_to_treemap_df(payload, doc_value=1, cluster_value_mode="remainder",
                        value_transform="sqrt",
                        value_floor=2.0,):
    """
    Convert LinkageClusterPacker payload with a single `root` node into a dataframe
    suitable for Plotly treemap rendering.

    Parameters
    ----------
    payload : dict
        Output from LinkageClusterPacker.build_payload()
    doc_value : int or float
        Value/area assigned to each document leaf.
    cluster_value_mode : str
        "remainder" -> cluster node value is sum(children/docs); use branchvalues="remainder"
        "total"     -> cluster node value is n_total_docs; use branchvalues="total"

    Returns
    -------
    pd.DataFrame
        Columns: id, parent, label, value, kind, level, n_total_docs, visible_n_docs
    """
    rows = []

    root = payload["root"]

    def transform_value(x):
        x = max(float(x), 1.0)
        if value_transform == "raw":
            return x
        elif value_transform == "sqrt":
            return value_floor + np.sqrt(x)
        elif value_transform == "log":
            return value_floor + np.log1p(x)
        else:
            raise ValueError("value_transform must be one of: raw, sqrt, log")

    def add_node(node, parent_id=""):
        node_id = str(node["cluster_id"])
        level = int(node.get("level", 0))
        raw_label = str(node.get("display_label", node["cluster_id"]))
        label = wrap_treemap_label_by_level(raw_label, level) #wrap_treemap_label(raw_label, width=14)

        if cluster_value_mode == "total":
            node_value = transform_value(node.get("n_total_docs", 0))
        else:
            node_value = 0

        rows.append({
            "id": node_id,
            "parent": parent_id,
            "label": label,
            "value": node_value,
            "kind": "cluster",
            "level": int(node.get("level", 0)),
            "n_total_docs": int(node.get("n_total_docs", 0)),
            "visible_n_docs": int(node.get("visible_n_docs", 0)),
        })

        # add document leaves
        doc_ids = node.get("documents", [])
        for doc in doc_ids:
            doc_node_id = f"{node_id}::doc::{doc}"
            rows.append({
                "id": doc_node_id,
                "parent": node_id,
                "label": wrap_treemap_label(str(doc), width=14),
                "value": float(max(doc_value, 1.0)),
                "kind": "document",
                "level": int(node.get("level", 0)) + 1,
                "n_total_docs": 1,
                "visible_n_docs": 1,
            })

        # recurse into children
        for child in node.get("children", []):
            add_node(child, parent_id=node_id)

    add_node(root, parent_id="")

    df = pd.DataFrame(rows)

    if cluster_value_mode == "remainder":
        # compute node values bottom-up as sum of immediate children's values
        value_map = {}

        children_map = {}
        for _, row in df.iterrows():
            children_map.setdefault(row["parent"], []).append(row["id"])

        kind_map = dict(zip(df["id"], df["kind"]))

        def compute_value(node_id):
            if kind_map[node_id] == "document":
                return float(df.loc[df["id"] == node_id, "value"].iloc[0])
        
            child_ids = children_map.get(node_id, [])
            total = 0.0
            for cid in child_ids:
                total += compute_value(cid)
        
            transformed = transform_value(total)
            value_map[node_id] = transformed
            return transformed

        compute_value(root["cluster_id"])

        df["value"] = df.apply(
            lambda r: value_map.get(r["id"], r["value"]),
            axis=1
        )

    return df

import colorsys
import plotly.graph_objects as go
import plotly.express as px


import plotly.express as px

def lighten_color(color, amount=0.0):
    if color.startswith("rgb"):
        vals = color[color.find("(")+1:color.find(")")].split(",")
        r, g, b = [int(v.strip()) for v in vals]
    else:
        color = color.lstrip("#")
        r = int(color[0:2], 16)
        g = int(color[2:4], 16)
        b = int(color[4:6], 16)

    r = int(r + (255 - r) * amount)
    g = int(g + (255 - g) * amount)
    b = int(b + (255 - b) * amount)

    return f"rgb({r},{g},{b})"


def build_focus_colors(df, focus_id="root", color_documents=True):
    palette = (
        px.colors.qualitative.Set3
        + px.colors.qualitative.Bold
        + px.colors.qualitative.Dark24
    )

    id_to_parent = dict(zip(df["id"], df["parent"]))
    id_to_level = dict(zip(df["id"], df["level"]))
    id_to_kind = dict(zip(df["id"], df["kind"]))

    focus_level = id_to_level.get(focus_id, 0)

    # Children of currently selected/focused node get distinct base colors
    focus_children = df.loc[df["parent"] == focus_id, "id"].tolist()

    base_color = {
        node_id: palette[i % len(palette)]
        for i, node_id in enumerate(focus_children)
    }

    def child_under_focus(node_id):
        """
        Find the first ancestor below focus_id.
        Example:
        focus_id = root.1
        node_id = root.1.2.3
        returns root.1.2
        """
        cur = node_id
        parent = id_to_parent.get(cur)

        while parent not in ("", None):
            if parent == focus_id:
                return cur
            cur = parent
            parent = id_to_parent.get(cur)

        return None

    colors = []

    for _, row in df.iterrows():
        node_id = row["id"]
        level = row["level"]
        kind = row["kind"]

        if node_id == focus_id:
            colors.append("lightgray")
            continue

        base_node = child_under_focus(node_id)

        # Outside the focused subtree
        if base_node is None:
            colors.append("rgba(230,230,230,0.35)")
            continue

        if color_documents and kind == "document":
            colors.append("rgba(220,220,220,0.65)")
            continue

        base = base_color.get(base_node, "#999999")

        # children of focus are full color; deeper descendants get lighter
        depth_below_focus_child = max(level - focus_level - 1, 0)
        lighten_amount = min(0.16 * depth_below_focus_child, 0.78)

        colors.append(lighten_color(base, lighten_amount))

    return colors

    
def render_payload_treemap(
    payload,
    doc_value=1,
    cluster_value_mode="remainder",
    width=1100,
    height=700,
    maxdepth=3,
    color_documents=True,
    focus_id="root",
):
    """
    Render payload as an inline Plotly treemap in Jupyter.

    Click rectangles to zoom in; use Plotly's path bar to go back up.
    """
    df = payload_to_treemap_df(
        payload,
        doc_value=doc_value,
        cluster_value_mode=cluster_value_mode,
    )

    branchvalues = "remainder" if cluster_value_mode == "remainder" else "total"

    if color_documents:

        colors = build_focus_colors(
            df,
            focus_id=focus_id,
            color_documents=color_documents,
        )
    else:
        colors = [f"level_{lvl}" for lvl in df["level"]]

    fig = go.Figure(go.Treemap(
        ids=df["id"],
        labels=df["label"],
        parents=df["parent"],
        values=df["value"],
        branchvalues=branchvalues,
        maxdepth=maxdepth,
        textinfo="label",
        hovertemplate=(
            "<b>%{label}</b><br>"
            "id=%{id}<br>"
            "value=%{value}<br>"
            "<extra></extra>"
        ),
        marker=dict(
            colors=colors,
            line=dict(width=1, color="white"),
        ),
        root_color="lightgray",
        pathbar=dict(visible=True),
    ))

    fig.update_traces(
        texttemplate="%{label}",
        textposition="middle center",
        tiling=dict(pad=1, packing="binary"), #packing: slice nice for longer titles, binary good balance with infill
        textfont_size=14,
    )

    fig.update_layout(
        width=width,
        height=height,
        margin=dict(t=50, l=2, r=2, b=2),
        title="Cluster Treemap",
        uniformtext=dict(minsize=10),#, mode="hide"),
    )

    fig.show()
    return df, fig

In [100]:
import plotly.io as pio
pio.renderers.default = "iframe"

In [102]:
df_tree, fig = render_payload_treemap(
    payload,
    doc_value=1,
    cluster_value_mode="remainder",
    width=1200,
    height=600,
    maxdepth=2
)

In [19]:
treemap_df = payload_to_treemap_df(payload, doc_value=1, cluster_value_mode="remainder")

In [20]:
treemap_df.head()

,id,parent,label,value,kind,level,n_total_docs,visible_n_docs
0,root,,root,164.0,cluster,0,164,7
1,root::doc::doc_162,root,doc_162,1.0,document,1,1,1
2,root::doc::doc_163,root,doc_163,1.0,document,1,1,1
3,root.1,root,root.1,130.0,cluster,1,130,3
4,root.1.1,root.1,root.1.1,107.0,cluster,2,107,7


In [21]:
type(fig)

plotly.graph_objs._figure.Figure